In [2]:
import os
import pandas as pd
from urllib.request import urlopen
from datetime import datetime

if not os.path.exists('data'):
    os.makedirs('data')

def download_data():
    base_url = "https://www.star.nesdis.noaa.gov/smcd/emb/vci/VH/get_TS_admin.php?country=UKR&provinceID={}&year1=1981&year2=2024&type=Mean"
    
    for i in range(1, 28):
        files = [f for f in os.listdir('data') if f.startswith(f'vhi_id_{i}_')]
        if files:
            print(f"Дані для області {i} вже завантажені.")
            continue
            
        try:
            url = base_url.format(i)
            with urlopen(url) as response:
                content = response.read()
                
            timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
            filename = f"data/vhi_id_{i}_{timestamp}.csv"
            
            with open(filename, 'wb') as f:
                f.write(content)
            print(f"Успішно завантажено область {i} -> {filename}")
        except Exception as e:
            print(f"Помилка при завантаженні області {i}: {e}")

download_data()

Дані для області 1 вже завантажені.
Дані для області 2 вже завантажені.
Дані для області 3 вже завантажені.
Дані для області 4 вже завантажені.
Дані для області 5 вже завантажені.
Дані для області 6 вже завантажені.
Дані для області 7 вже завантажені.
Дані для області 8 вже завантажені.
Дані для області 9 вже завантажені.
Дані для області 10 вже завантажені.
Дані для області 11 вже завантажені.
Дані для області 12 вже завантажені.
Дані для області 13 вже завантажені.
Дані для області 14 вже завантажені.
Дані для області 15 вже завантажені.
Дані для області 16 вже завантажені.
Дані для області 17 вже завантажені.
Дані для області 18 вже завантажені.
Успішно завантажено область 19 -> data/vhi_id_19_20260423_234459.csv
Успішно завантажено область 20 -> data/vhi_id_20_20260423_234501.csv
Успішно завантажено область 21 -> data/vhi_id_21_20260423_234503.csv
Успішно завантажено область 22 -> data/vhi_id_22_20260423_234505.csv
Успішно завантажено область 23 -> data/vhi_id_23_20260423_234511.cs

In [3]:
def clean_and_merge(folder_path):
    noaa_to_ua = {
        1: ("Черкаська", 22), 2: ("Чернігівська", 24), 3: ("Чернівецька", 23),
        4: ("Крим", 25), 5: ("Дніпропетровська", 3), 6: ("Донецька", 4),
        7: ("Івано-Франківська", 8), 8: ("Харківська", 19), 9: ("Херсонська", 20),
        10: ("Хмельницька", 21), 11: ("Київська", 9), 12: ("Київ", 26),
        13: ("Кіровоградська", 10), 14: ("Луганська", 11), 15: ("Львівська", 12),
        16: ("Миколаївська", 13), 17: ("Одеська", 14), 18: ("Полтавська", 15),
        19: ("Рівненська", 16), 20: ("Севастополь", 27), 21: ("Сумська", 17),
        22: ("Тернопільська", 18), 23: ("Вінницька", 1), 24: ("Волинська", 2),
        25: ("Закарпатська", 6), 26: ("Запорізька", 7), 27: ("Житомирська", 5)
    }

    all_data = []
    for filename in os.listdir(folder_path):
        if filename.endswith(".csv"):
            orig_id = int(filename.split('_')[2])
            path = os.path.join(folder_path, filename)
            
            df = pd.read_csv(path, index_col=False, header=1, names=['Year', 'Week', 'SMN', 'SMT', 'VCI', 'TCI', 'VHI', 'junk'])
            df = df.drop(columns=['junk'])
            df = df.dropna()
            
            df['Year'] = df['Year'].astype(str).str.replace('<tt><pre>', '', regex=False).astype(int)
            
            name, new_id = noaa_to_ua[orig_id]
            df['Province_Name'] = name
            df['Province_Index'] = new_id
            
            all_data.append(df)
            
    return pd.concat(all_data, ignore_index=True)

df = clean_and_merge('data')
df['Week'] = df['Week'].astype(int)
print("Дані успішно підготовлені. Перші 5 рядків:")
display(df.head())

Дані успішно підготовлені. Перші 5 рядків:


,Year,Week,SMN,SMT,VCI,TCI,VHI,Province_Name,Province_Index
0,1982,1,0.067,266.68,41.66,37.50,39.58,Одеська,14
1,1982,2,0.064,267.80,41.90,34.64,38.27,Одеська,14
2,1982,3,0.061,268.21,40.71,33.80,37.26,Одеська,14
3,1982,4,0.055,268.14,35.53,35.58,35.55,Одеська,14
4,1982,5,0.048,268.13,30.39,37.23,33.81,Одеська,14


In [4]:
def get_vhi_by_year(df, province_id, year):
    result = df[(df['Province_Index'] == province_id) & (df['Year'] == year)][['Week', 'VHI']]
    print(f"VHI для області {province_id} за {year} рік:")
    return result

def get_vhi_by_range(df, province_ids, start_year, end_year):
    result = df[(df['Province_Index'].isin(province_ids)) & 
                (df['Year'] >= start_year) & 
                (df['Year'] <= end_year)][['Province_Name', 'Year', 'Week', 'VHI']]
    return result

def get_vhi_stats(df, province_id, year_range):
    subset = df[(df['Province_Index'] == province_id) & 
                (df['Year'] >= year_range[0]) & (df['Year'] <= year_range[1])]
    
    stats = {
        'Min VHI': subset['VHI'].min(),
        'Max VHI': subset['VHI'].max(),
        'Mean VHI': subset['VHI'].mean(),
        'Median VHI': subset['VHI'].median()
    }
    return pd.Series(stats)

display(get_vhi_by_year(df, 1, 2020).head())
print("\nСтатистика для Вінницької області (ID 1) за 2010-2020:")
display(get_vhi_stats(df, 1, (2010, 2020)))

VHI для області 1 за 2020 рік:


,Week,VHI
57876,1,39.29
57877,2,39.08
57878,3,38.94
57879,4,39.47
57880,5,40.11



Статистика для Вінницької області (ID 1) за 2010-2020:


Min VHI       31.600000
Max VHI       69.280000
Mean VHI      49.707902
Median VHI    49.515000
dtype: float64